# Anchor extraction demo

Core engine: the LLM returns **verbatim boundary anchors only**; Python resolves offsets, slices text, groups segments by role, and stitches across chunks.

An **extraction profile** (detection prompt + optional unit-boundary regex) decides which logical units to extract. Bundled profiles: `anchor_extract/prompts/profiles/`.

The LLM provider (Azure OpenAI or Anthropic) is chosen from `.env` (`ANCHOR_LLM_PROVIDER=auto`).

Set `PDF_PATH` to a text-extractable PDF. Docs: `docs/core_pipeline.md`, `docs/profiles.md`.

**Phased flow** (see `docs/phases.md`):

- **Phase 1 — read:** `anchor.read(PDF_PATH, ...)` returns a `Document` over the positional block model (PDF or `.txt`).
- **Phase 2 — blocks:** `doc.blocks()` renders that document as JSON for inspection (`anchor.blocks(doc)` is the equivalent module-level form).
- **Phase 3 — extract:** `extract_document(..., doc=doc.extraction)` runs the anchor contract against the LLM.

PDF extraction drops running headers/footers, page numbers, and repeated header lines before `full_text` is built.

Larger `END_PAGE` needs a unit-boundary regex (`EXAMPLE_HIPAA_SECTION_BOUNDARY_PATTERN`) so batches stay one (or few) CFR sections — otherwise the model omits units such as § 160.103.

Chunk/batch sweet-spot experiments (offline, no LLM): `python scripts/sweep_chunk_budget.py --pdf pdf/hipaa-simplification-201303.pdf --start-page 11 --end-page 25`. See `docs/testing.md`.


In [1]:
import json
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv

import anchor
from anchor_extract import build_anchor_system_prompt
from anchor_extract.pipeline import extract_document, to_requirements_json, save_json
from anchor_extract.settings import EXAMPLE_AI_RMF_BOUNDARY_PATTERN, EXAMPLE_HIPAA_SECTION_BOUNDARY_PATTERN

load_dotenv()

# HIPAA (matches PDF_PATH)
PDF_PATH = Path("pdf/hipaa-simplification-201303.pdf")
DETECTION_PROMPT_PATH = Path("anchor_extract/prompts/profiles/hipaa.txt")
BOUNDARY_PATTERN = EXAMPLE_HIPAA_SECTION_BOUNDARY_PATTERN

# Alternative: NIST AI RMF Playbook
# PDF_PATH = Path("pdf/AI_RMF_Playbook.pdf")
# DETECTION_PROMPT_PATH = Path("anchor_extract/prompts/profiles/ai_rmf_playbook.txt")
# BOUNDARY_PATTERN = EXAMPLE_AI_RMF_BOUNDARY_PATTERN


## Phase 1 — Read

`anchor.read` dispatches on the file suffix: `.pdf` goes through the PyMuPDF extractor, `.txt` through the plain-text ingest (page arguments ignored). Both return a `Document`, whose engine-level extraction stays reachable through `doc.extraction`.

In [2]:
START_PAGE = 11
END_PAGE = 17

doc = anchor.read(path=PDF_PATH,
                  start_page=START_PAGE,
                  end_page=END_PAGE)

print("pages:", doc.extraction.start_page, "-", doc.extraction.end_page,
      "| blocks:", len(doc.extraction.blocks), "| chars:", len(doc.full_text))

pages: 11 - 17 | blocks: 199 | chars: 26241


## Phase 2 — Blocks JSON

`doc.blocks()` is the JSON-serializable view: `schema_version`, `source`, `summary` and one record per block (`full_text` omitted unless `include_full_text=True`). `anchor.blocks(doc)` returns the same dict.

The block objects themselves live on `doc.extraction.blocks`.

In [3]:
blocks_payload = doc.blocks()
# Same records as blocks_payload["blocks"], first 15 for readability.
blocks_df = pd.DataFrame([{
    "page": b.page,
    "char_start": b.char_start,
    "char_end": b.char_end,
    "text_preview": b.text[:120],
} for b in doc.extraction.blocks[:15]])
blocks_df

,page,char_start,char_end,text_preview
0,11,0,24,§ 160.102 Applicability.
1,11,25,184,"(a) Except as otherwise provided, the standard..."
2,11,185,203,(1) A health plan.
3,11,204,236,(2) A health care clearinghouse.
4,11,237,380,(3) A health care provider who transmits any h...
5,11,381,524,"(b) Where provided, the standards, requirement..."
6,11,525,809,(c) To the extent required under the Social Se...
7,11,810,907,"[65 FR 82798, Dec. 28, 2000, as amended at 67 ..."
8,11,908,930,§ 160.103 Definitions.
9,11,931,1012,"Except as otherwise provided, the following de..."


## Phase 3 — Extract

The extraction profile is loaded, composed with the generic anchor contract, and run over the document read in phase 1. This is the only step that calls the LLM.

In [4]:
detection_prompt = DETECTION_PROMPT_PATH.read_text(encoding="utf-8")
system_prompt = build_anchor_system_prompt(detection_prompt)

if system_prompt:
    print("Prompt loaded correctly")
else:
    print("Error loading prompt")

extraction = extract_document(
    str(PDF_PATH),
    detection_prompt,
    doc=doc.extraction,
    start_page=START_PAGE,
    end_page=END_PAGE,
    requirement_boundary_pattern=BOUNDARY_PATTERN,
    verbose=True,)

Prompt loaded correctly
Chunk budget: 12000 input tok (override; ctx 200000, prompt~4068, out 8000)
Boundary-aware chunking: 5 unit start(s) detected in blocks 0-198
[iter   1] blocks    0-198  | pages  11-17  | est  6560 tok | target 12000 | shrinks 0 ... 5 reqs (5 ok, 0 trunc) | stop=tool_use | 10.509s
  -> stitched 4 requirement(s) | pending OPEN
  ! range ended with unit #5 (a322e3195123:26004) still open; emitting bounded (end_resolved=False)

Done. 1 batch(es) in 10.6s | 1/1 ok API call(s) | 5 requirement(s) stitched (4 flagged).


In [5]:
results_df = pd.DataFrame([{
    "anchor_id": r.anchor_id,
    "span_key": r.span_key,
    "status": r.status,
    "verbatim_match": r.verbatim_match,
    "end_resolved": r.end_resolved,
    "n_segments": r.n_segments,
    "n_segments_resolved": r.n_segments_resolved,
    "doc_offset_start": r.doc_offset_start,
    "doc_offset_end": r.doc_offset_end,
} for r in extraction.requirements])

results_df

,anchor_id,span_key,status,verbatim_match,end_resolved,n_segments,n_segments_resolved,doc_offset_start,doc_offset_end
0,1,a322e3195123:0,complete,False,False,1,1,0,908
1,2,a322e3195123:908,complete,True,True,1,1,908,24093
2,3,a322e3195123:24094,complete,False,False,1,1,24094,25273
3,4,a322e3195123:25273,complete,False,False,1,1,25273,26004
4,5,a322e3195123:26004,truncated_at_end,False,False,1,1,26004,26239


In [6]:
req_id = "1"

for i in extraction.requirements:
    if i.anchor_id == req_id:
        print(i.original_text)
        print("-" * 100)
        print(f"start: {i.start_anchor}")
        print(f"end: {i.end_anchor}")

§ 160.102 Applicability.
(a) Except as otherwise provided, the standards, requirements, and implementation specifications adopted under this subchapter apply to the following entities:
(1) A health plan.
(2) A health care clearinghouse.
(3) A health care provider who transmits any health information in electronic form in connection with a transaction covered by this subchapter.
(b) Where provided, the standards, requirements, and implementation specifications adopted under this subchapter apply to a business associate.
(c) To the extent required under the Social Security Act, 42 U.S.C. 1320a-7c(a)(5), nothing in this subchapter shall be construed to diminish the authority of any Inspector General, including such authority as provided in the Inspector General Act of 1978, as amended (5 U.S.C. App.).
[65 FR 82798, Dec. 28, 2000, as amended at 67 FR 53266, Aug. 14, 2002; 78 FR 5687, Jan. 25, 2013]

-------------------------------------------------------------------------------------------

In [7]:
from anchor_extract import inspect_requirement, unit_metadata

view = inspect_requirement(extraction, "1", doc=doc.extraction)
print(view["metadata"]["unit"].get("req_id"))
print(view["model_anchors"]["status"], view["model_anchors"]["start_anchor"])
print([call["chunk_id"] for call in view["llm_calls"]])

for r in extraction.requirements:
    printed = unit_metadata(r).get("req_id")
    if r.n_segments != 1 or r.doc_offset_start < 0:
        print(r.anchor_id, printed, "invariant: n/a (multi-span or unresolved)")
        continue
    ok = doc.full_text[r.doc_offset_start:r.doc_offset_end] == r.original_text
    print(r.anchor_id, printed, "invariant:", "ok" if ok else "BAD")

160.102
complete § 160.102 Applicability.
(a) Except as otherwise provided, the standards, requirements, and implementation specifications adopted under this subchapter apply to the following entities:
(1) A health plan.
(2) A health care clearinghouse.
(3) A health care provider who transmits any health information in electronic form in connection with a transaction covered by this subchapter.
(b) Where provided, the standards, requirements, and implementation specifications adopted under this subchapter apply to a business associate.
(c) To the extent required under the Social Security Act, 42 U.S.C. 1320a-7c(a)(5), nothing in this subchapter shall be construed to diminish the authority of any Inspector General, including such authority as provided in the Inspector General Act of 1978, as amended (5 U.S.C. App.).
[65 FR 82798, Dec. 28, 2000, as amended at 67 FR 53266, Aug. 14, 2002; 78 FR 5687, Jan. 25, 2013]
[1]
1 160.102 invariant: ok
2 160.103 invariant: ok
3 160.104 invariant: ok